#Get Test Sequences:

In [ ]:
"""
Fetch full-length protein sequences from UniProt or dbPTM given IDs from
Succinylation_pos.fasta and Succinylation_neg.fasta. The IDs in these
FASTA headers are of the form >12345_SPECIES_678, and we use the
12345_SPECIES part as the key ID to query UniProt or dbPTM.
"""

import requests
from Bio import SeqIO
from bs4 import BeautifulSoup
import sys

# File names for positive and negative FASTA input
pos_fasta = "DbPtm/Succinylation_pos.fasta"
neg_fasta = "DbPtm/Succinylation_neg.fasta"
output_fasta = "DbPtm/DbPtm_full_sequences.fasta"

# Extract dbPTM ID (first and second parts)
def extract_id_from_header(header):
    parts = header.lstrip(">").split("_")
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1]}"
    return None

# Parse all headers to get unique dbPTM IDs
protein_ids = set()
for fasta_file in [pos_fasta, neg_fasta]:
    try:
        for record in SeqIO.parse(fasta_file, "fasta"):
            pid = extract_id_from_header(record.id)
            if pid:
                protein_ids.add(pid)
    except FileNotFoundError:
        sys.exit(f"Error: File {fasta_file} not found.")

print(f"Found {len(protein_ids)} unique protein IDs to fetch.")

# UniProt fetch
def fetch_sequence_from_uniprot(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
    try:
        res = requests.get(url, timeout=10)
    except requests.RequestException as e:
        print(f"Warning: UniProt request failed for {uniprot_id}: {e}")
        return None, None, None

    if res.status_code == 200:
        lines = res.text.strip().splitlines()
        if len(lines) >= 2:
            header = lines[0]
            seq = "".join(lines[1:])
            ac = header.split("|")[1] if "|" in header else uniprot_id
            return header, seq, ac
    return None, None, None

# dbPTM fallback
def fetch_sequence_from_dbptm(protein_id):
    url = f"https://biomics.lab.nycu.edu.tw/dbPTM/info.php?id={protein_id}"
    try:
        res = requests.get(url, timeout=10)
    except requests.RequestException as e:
        print(f"Warning: dbPTM request failed for {protein_id}: {e}")
        return None, None, None

    if res.status_code == 200:
        soup = BeautifulSoup(res.text, "html.parser")

        sequence = None
        uniprot_ac = None

        for row in soup.find_all("tr"):
            header_cell = row.find("th")
            value_cell = row.find("td")
            if not header_cell or not value_cell:
                continue

            label = header_cell.get_text(strip=True)
            if label == "Protein Sequence":
                sequence = value_cell.get_text(strip=True)
            elif label == "UniProt AC":
                uniprot_ac = value_cell.get_text(strip=True)

        if sequence:
            ac = uniprot_ac if uniprot_ac else protein_id
            header = f">{ac}_{protein_id}"
            return header, sequence, ac

    return None, None, None

# Fetch and write
seen_ac = set()
written = 0
skipped = 0

with open(output_fasta, "w") as out_f:
    for uid in sorted(protein_ids):
        # Try UniProt
        header, sequence, ac = fetch_sequence_from_uniprot(uid)

        # Fallback to dbPTM
        if not header or not sequence:
            _, _, ac = fetch_sequence_from_dbptm(uid)     
            header, sequence, ac = fetch_sequence_from_uniprot(ac)


        if header and sequence and ac:
            if ac not in seen_ac:
                seen_ac.add(ac)
                out_f.write(f"{header}\n")
                out_f.write(f"{sequence}\n")
                written += 1
            else:
                print(f"Info: ID {ac} already written, skipping duplicate.")
                skipped += 1
        else:
            print(f"Warning: No sequence retrieved for ID {uid}")
            skipped += 1

        print(f"Written: {written}, Skipped : {skipped}, Current ID: {uid}")

print(f"\n✅ Done! Sequences written to {output_fasta}")


Found 4275 unique protein IDs to fetch.
Written: 1, Skipped : 0, Current ID: 14331_SOLLC
Written: 2, Skipped : 0, Current ID: 14333_ORYSJ
Written: 3, Skipped : 0, Current ID: 14334_ORYSJ
Written: 4, Skipped : 0, Current ID: 1433B_HUMAN
Written: 5, Skipped : 0, Current ID: 1433E_HUMAN
Written: 6, Skipped : 0, Current ID: 1433T_HUMAN
Written: 7, Skipped : 0, Current ID: 2AAA_HUMAN
Written: 8, Skipped : 0, Current ID: 35KD_MYCTU
Written: 9, Skipped : 0, Current ID: 3BHS5_MOUSE
Written: 10, Skipped : 0, Current ID: 3HIDH_HUMAN
Written: 11, Skipped : 0, Current ID: 3HIDH_MOUSE
Written: 12, Skipped : 0, Current ID: 3PASE_ECOLI
Written: 13, Skipped : 0, Current ID: 40C1_ORYSJ
Written: 14, Skipped : 0, Current ID: 4F2_HUMAN
Written: 15, Skipped : 0, Current ID: 5NT3A_MOUSE
Written: 16, Skipped : 0, Current ID: 6PGD1_YEAST
Written: 17, Skipped : 0, Current ID: 6PGD_ECOLI
Written: 18, Skipped : 0, Current ID: 6PGD_MOUSE
Written: 19, Skipped : 0, Current ID: 6PGL_ECOLI
Written: 20, Skipped : 0, C

In [25]:
def parse_fasta(filename):
    entries = {}
    with open(filename, "r") as f:
        lines = f.readlines()
    current_id = None
    sequence_parts = []
    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            if current_id is not None:
                entries[current_id] = ''.join(sequence_parts)
            if "|" in line:
                current_id = line.split("|")[-1]
            else:
                current_id = line[1:]
            sequence_parts = []
        else:
            sequence_parts.append(line)
    if current_id:
        entries[current_id] = ''.join(sequence_parts)
    return entries

def parse_fasta_train(filename):
    entries = {}
    with open(filename, "r") as f:
        lines = f.readlines()
    current_id = None
    sequence_parts = []
    for line in lines:
        line = line.strip()
        if line.startswith(">"):
            if current_id is not None:
                entries[current_id] = ''.join(sequence_parts)
            # Parse ID from header like '>tr|A0A087WQC8|A0A087WQC8_MOUSE'
            parts = line.split("|")
            if len(parts) >= 2:
                current_id = parts[1]
            else:
                current_id = line[1:]
            sequence_parts = []
        else:
            sequence_parts.append(line)
    if current_id:
        entries[current_id] = ''.join(sequence_parts)
    return entries

In [ ]:
# Load both files
file1 = parse_fasta("DbPtm/Succinylation_neg.fasta")
file2 = parse_fasta("DbPtm/DbPtm_full_sequences.fasta")

matched = 0
unmatched = 0
not_found = 0

with open("DbPtm\Succinylation_neg_matched.fasta", "w") as matched_file:
    for file1_id, peptide in file1.items():
        base_id = "_".join(file1_id.split("_")[:2])
        found = False
        for file2_id, full_seq in file2.items():
            if base_id in file2_id:
                found = True
                if peptide in full_seq:
                    matched += 1
                    matched_file.write(f">{file1_id}\n{peptide}\n")
                else:
                    unmatched += 1
                break
        if not found:
            not_found += 1

print("Matched:", matched)
print("Unmatched:", unmatched)
print("ID Not Found:", not_found)


In [2]:
import h5py
import csv
import json
import re
from Bio import SeqIO

# Load full sequences from the FASTA file
def load_full_sequences(fasta_path):
    sequences = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        header = record.id  # e.g., sp|P93206|14331_SOLLC
        protein_id = header.split("|")[2]  # e.g., 14331_SOLLC
        sequences[protein_id] = str(record.seq)
    return sequences

# Parse site-specific FASTA to extract (protein_id, site, sequence)
def parse_site_fasta(fasta_path, label):
    entries = []
    with open(fasta_path, 'r') as f:
        lines = f.read().splitlines()
    for i in range(0, len(lines), 2):
        header = lines[i].lstrip(">")
        sequence = lines[i + 1]
        protein_id, site = header.rsplit("_", 1)
        entries.append((protein_id, int(site), sequence, label))
    return entries

# Main CSV creation function
def create_benchmark_csv(pos_path, neg_path, full_seq_path, h5_path, output_csv):
    full_sequences = load_full_sequences(full_seq_path)
    entries = parse_site_fasta(pos_path, label=1) + parse_site_fasta(neg_path, label=0)

    with h5py.File(h5_path, 'r') as h5_file, open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['protein_id', 'site', 'sequence', 'embedding', 'label', 'full_sequence'])

        for protein_id, site, seq21, label in entries:
            print(f"Processing {protein_id} at site {site} with label {label}")
            if protein_id not in full_sequences:
                print(f"⚠️ Full sequence for {protein_id} not found. Skipping.")
                continue

            full_seq = full_sequences[protein_id]
            start = max(0, site - 17)
            end = min(len(full_seq), site + 16)
            seq33 = full_seq[start:end]
            # Pad with '-' if out of bounds
            left_pad = max(0, 16 - (site - 1))
            right_pad = max(0, (site + 16) - len(full_seq))
            seq33 = ('-' * left_pad) + seq33 + ('-' * right_pad)
            # Find the index of seq21 in full_seq and check if it's centered in seq33
            try:
                full_index = full_seq.index(seq21) + (len(seq21) // 2)
            except ValueError:
                print(f"⚠️ Sequence '{seq21}' not found in full sequence of {protein_id}. Skipping.")
                continue

            # Check if seq21 is in the middle of seq33
            center_start = (len(seq33) - len(seq21)) // 2
            center_seq = seq33[center_start:center_start + len(seq21)]
            if center_seq != seq21:
                print(f"⚠️ seq21 is not centered in seq33 for {protein_id} at site {site}. Skipping.")
                print(f"    seq21: XXXXXX{seq21}XXXXXX\n    seq33: {seq33}")
                continue

            # Match the full UniProt ID from h5
            matched_key = next((k for k in h5_file.keys() if protein_id in k), None)
            if not matched_key:
                print(f"⚠️ Embedding for {protein_id} not found in H5. Skipping.")
                continue
            print(f"Matched key: {matched_key}")
            embedding_array = h5_file[matched_key][()]  # Shape: (len+2, 1024)
            protT5_index = site + 2  # Account for special token at start

            if protT5_index >= embedding_array.shape[0]:
                print(f"⚠️ Index out of bounds for {protein_id}. Skipping.")
                continue

            site_embedding = embedding_array[protT5_index].tolist()

            writer.writerow([protein_id, site, seq33, json.dumps(site_embedding), label, full_seq])

    print(f"✅ Benchmark CSV saved to: {output_csv}")


create_benchmark_csv(
    pos_path="DbPtm/Succinylation_pos_matched.fasta",
    neg_path="DbPtm/Succinylation_neg_matched.fasta",
    full_seq_path="DbPtm/dbptm_test_filtered.fasta",
    h5_path="Embeddings/DBPTM_embeddings.h5",
    output_csv="DbPtm/benchmark_filtered_with_full_sequence.csv"
)


Processing 14331_SOLLC at site 130 with label 1
⚠️ Full sequence for 14331_SOLLC not found. Skipping.
Processing 14333_ORYSJ at site 70 with label 1
⚠️ Full sequence for 14333_ORYSJ not found. Skipping.
Processing 14334_ORYSJ at site 146 with label 1
⚠️ Full sequence for 14334_ORYSJ not found. Skipping.
Processing 1433B_HUMAN at site 159 with label 1
⚠️ Full sequence for 1433B_HUMAN not found. Skipping.
Processing 1433E_HUMAN at site 28 with label 1
⚠️ Full sequence for 1433E_HUMAN not found. Skipping.
Processing 1433E_HUMAN at site 50 with label 1
⚠️ Full sequence for 1433E_HUMAN not found. Skipping.
Processing 1433T_HUMAN at site 80 with label 1
⚠️ Full sequence for 1433T_HUMAN not found. Skipping.
Processing 2AAA_HUMAN at site 542 with label 1
⚠️ Full sequence for 2AAA_HUMAN not found. Skipping.
Processing 3BHS5_MOUSE at site 175 with label 1
⚠️ Full sequence for 3BHS5_MOUSE not found. Skipping.
Processing 3BHS5_MOUSE at site 203 with label 1
⚠️ Full sequence for 3BHS5_MOUSE not fou